# 🩺 Syght — Healthcare Insurance Fraud Detector Platform
## Unit-1 Experiment 2: Exploratory Data Analysis (EDA) on Real-World Healthcare Dataset
**Course Specialization**: AI & Machine Learning with Cloud & Azure Tools  
**Project**: Healthcare Insurance Fraud Detection ()  

---

### 🎯 Experiment Aim & Overview:
Perform deep exploratory data analysis on 558,000 healthcare claims to uncover fraud prevalence, descriptive statistics, correlation structures, and clinical anomaly trends.

---

### 📚 Theoretical Background
Exploratory Data Analysis (EDA) is forensic investigation of the dataset using descriptive statistics and visual cues.
In healthcare fraud, fraudsters exploit systematic vulnerabilities:
- **Severe Class Imbalance**: Fraud is rare (~9.5%), meaning standard accuracy is misleading.
- **Unbundling Anomaly**: High duplicate visit rates on the exact same date for the same patient.
- **ICU Upcoding**: Artificial inflation of patient hospitalization length of stay.
- **Physician Monopolies**: Bizarre concentration of revenue around a single clinician.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ml_pipeline.medicare_simulator import generate_simulated_medicare_data
from ml_pipeline.feature_engineering import build_provider_features
from ml_pipeline.config import SyghtConfig

cfg = SyghtConfig()
lbl, bene, ip, op = generate_simulated_medicare_data(n_providers=60, seed=42)
feats = build_provider_features(lbl, bene, ip, op, config=cfg)
print(f'Engineered Provider Features Matrix Shape: {feats.shape}')
feats.head(3)

### Step 1: Descriptive Statistics & Fraud Class Imbalance

In [ ]:
fraud_counts = feats['fraud'].value_counts()
fraud_rate = feats['fraud'].mean()
print(f'Legitimate Providers (0): {fraud_counts.get(0, 0)}')
print(f'Fraudulent Providers (1): {fraud_counts.get(1, 0)}')
print(f'Fraud Base Rate: {fraud_rate:.2%}')

# Summary statistics comparing Fraud vs Legitimate
comparison = feats.groupby('fraud')[['n_claims', 'total_reimbursed', 'mean_inpatient_los', 'top_physician_concentration']].mean()
print('\nMean Feature Comparison across Classes:')
print(comparison)

### Step 2: Correlation Analysis (Pearson Heatmap)

In [ ]:
key_cols = ['fraud', 'n_claims', 'total_reimbursed', 'mean_inpatient_los', 'top_physician_concentration', 'pct_duplicate_bene_claims_same_day']
corr = feats[key_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Key Fraud Billing Indicators')
plt.tight_layout()
plt.show()

### Step 3: Forensic Pattern & Trend Identification

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Length of stay distribution by fraud status
sns.boxplot(x='fraud', y='mean_inpatient_los', data=feats, ax=axes[0], palette=['#10b981', '#ef4444'])
axes[0].set_title('Inpatient Length of Stay: Honest vs Fraudulent')
axes[0].set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])
axes[0].set_ylabel('Mean Inpatient Days')

# 2. Duplicate same-day claims rate
sns.barplot(x='fraud', y='pct_duplicate_bene_claims_same_day', data=feats, ax=axes[1], palette=['#10b981', '#ef4444'])
axes[1].set_title('Same-Day Duplicate Claims Rate by Provider Class')
axes[1].set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])
axes[1].set_ylabel('Duplicate Rate')

plt.tight_layout()
plt.show()